# 说明

这个单元就是把一些讲过的内容用Microsoft Agent Framework 框架实现一遍，Microsoft Agent Framework 是微软将AutoGen 和 Semantic Kernel (SK)这两个框架的主要优势合并，形成了一个统一的开源框架

## 两个顺序代理：

1. **前台代理**：提供城市的初步吸引力推荐  
2. **礼宾代理**：根据受欢迎程度审查并评分前台代理的推荐  

## 顺序编排的主要优势：

- **迭代优化**：第二个代理改进第一个代理的工作  
- **专业化**：每个代理在流程中都有特定的角色  
- **质量控制**：内置的审查和验证步骤  
- **信息流清晰**：代理之间结构化的交接  

## 前提条件：
- 安装 Microsoft Agent Framework  
- 配置 DashScope API key 或其他 OpenAI 兼容服务  
- 了解基本代理概念  

注意：如果你使用 `qwen-max`，建议使用 `OpenAIChatCompletionClient`，因为它走 `chat.completions` 接口，比 `OpenAIChatClient` 的 `responses` 接口更兼容 DashScope。


In [1]:
import sys
!{sys.executable} -m pip install agent-framework-core agent-framework-openai agent-framework-orchestrations



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# ===== 第一部分：导入必要的库 =====
# 异步编程支持（使代码能同时处理多任务）
import asyncio
# JSON数据处理
import json
# 操作系统功能（如环境变量）
import os
# 类型提示（帮助理解代码）
from typing import Any, cast

# Microsoft Agent Framework 当前版本核心组件
# 注意：新版本里 ChatMessage 改名为 Message，
# SequentialBuilder 位于 agent_framework.orchestrations 命名空间下，
# agent 需要通过 Agent(...) 显式创建。
from agent_framework import Agent, Message
from agent_framework.orchestrations import SequentialBuilder

# OpenAI 兼容客户端
# 这里使用 OpenAIChatCompletionClient，以兼容 DashScope 的 qwen-max。
from agent_framework.openai import OpenAIChatCompletionClient
# 从.env文件加载环境变量
from dotenv import load_dotenv
# 在Jupyter中显示HTML内容
from IPython.display import HTML, display
# 用于定义结构化数据模型
from pydantic import BaseModel, Field

print("All imports successful!")


All imports successful!


## 第一步：定义用于结构化输出的 Pydantic 模型

这些模型定义了每个代理将返回的模式。前台代理提供推荐，礼宾代理提供评论和评分。


In [3]:
# ===== 第二部分：定义结构化输出模型 =====
# 这些模型确保代理返回的数据格式正确、完整
# 注意：为了提高 gpt-4.1-mini 在结构化输出下的稳定性，
# 对少量容易偶发缺失的字段设置默认值，避免一次漏字段就导致整个工作流中断。
class AttractionRecommendation(BaseModel):
    """前台代理返回的景点推荐数据结构"""

    city: str  # 城市名称
    attraction_name: str  # 景点名称
    description: str  # 景点描述
    category: str  # 景点类别（如"博物馆"、"地标"等）
    recommended_duration: str  # 推荐游览时长（如"2-3小时"）
    why_recommended: str = Field(default="没有提供任何理由。")  # 推荐理由
    best_time_to_visit: str = Field(default="最佳时间未指定。")  # 最佳参观时间


class AttractionReview(BaseModel):
    """礼宾代理返回的景点评审数据结构"""

    attraction_name: str  # 景点名称
    city: str  # 城市名称
    popularity_score: int = Field(default=7)  # 流行度评分（1-10分）
    popularity_reasoning: str = Field(default="没有提供任何理由。")  # 评分理由
    visitor_rating: float = Field(default=4.0)  # 游客评分（1.0-5.0分）
    pros: list[str] = Field(default_factory=list)  # 优点列表
    cons: list[str] = Field(default_factory=list)  # 缺点列表
    concierge_recommendation: str = Field(default="没有提供任何建议。")  # 礼宾建议
    alternative_suggestions: list[str] = Field(default_factory=list)  # 替代建议


## 第2步：加载环境变量

下面使用 OpenAI 兼容方式配置聊天客户端。

如果你使用的是：

- GitHub Models：可以继续填写 `GITHUB_ENDPOINT / GITHUB_TOKEN`
- DashScope + qwen-max：建议填写 `DASHSCOPE_API_KEY`，并把 `base_url` 指向兼容接口

本 notebook 当前按 `DashScope + qwen-max` 方式演示。


In [4]:
# ===== 第三部分：配置LLM服务 =====

# Load environment variables
# 加载环境变量（从.env文件）
load_dotenv()

chat_client = OpenAIChatCompletionClient(
    base_url="https://models.inference.ai.azure.com/",  # DashScope OpenAI兼容接口
    api_key=os.environ.get("GITHUB_TOKEN"),                  # DashScope API Key
    model="gpt-4.1-mini"                                              # 使用的模型名称
)

print("Chat client configured successfully!")


Chat client configured successfully!


## 第三步：创建两个顺序代理

每个代理在顺序工作流程中都有特定的角色。前台代理负责提供推荐，而礼宾代理负责审核并评分这些推荐。


In [5]:
# ===== 第四部分：创建专业代理 =====
# 代理1：前台接待员（提供景点初步推荐）
# Agent 1: Front Desk Agent (Makes initial recommendations)
front_desk_agent = Agent(
    client=chat_client,
    instructions=(
        """
        你是一名知识丰富的酒店前台接待员，专门为住客推荐当地旅游景点。

        当住客询问某个城市有哪些值得游览的景点时，请为其推荐**一个**经过充分了解、最具代表性的热门旅游景点。

        你的推荐应重点提供以下实用信息：
        - 该景点有哪些独特之处；
        - 建议游览时长；
        - 最佳游览时间。

        请以热情、友好的态度为住客提供有帮助的推荐。

        **你必须仅返回合法的 JSON 格式，不要输出任何其他内容。**

        返回的 JSON 必须始终包含以下字段，并且字段名称必须完全一致：

        - city
        - attraction_name
        - description
        - category
        - recommended_duration
        - why_recommended
        - best_time_to_visit
        """
    ),
    name="front_desk_agent",
    default_options={"response_format": AttractionRecommendation},
)

# Agent 2: Concierge Agent (Reviews and rates recommendations)
# 代理2：礼宾（提供专家评审和评分）
# 指令中文翻译如下：
# 你是一个专家级礼宾，精通全球旅游景点。
# 你将收到一个景点推荐，并需要提供专家评审和评分。
# 根据景点的流行度、游客满意度和整体质量进行评估。  
# 提供流行度评分（1-10分）、游客评分（1.0-5.0分）、列出优缺点，并给出专业评估。
# 如有必要，还要建议替代景点。返回指定格式的JSON数据。
concierge_agent = Agent(
    client=chat_client,
    instructions=(
        """
        你是一名经验丰富的礼宾服务专家（Concierge），对全球各地的旅游景点都有深入的了解。

        你将收到一份旅游景点推荐，需要从专业角度对该推荐进行评估，并给出专家点评和评分。

        请根据以下几个方面进行综合评价：
        - 景点的知名度和受欢迎程度；
        - 游客满意度；
        - 景点的整体品质。

        请提供以下内容：
        - 景点受欢迎程度评分（1-10 分）；
        - 游客评分（1.0-5.0 分）；
        - 景点的优点（Pros）；
        - 景点的缺点（Cons）；
        - 你的专业评价与推荐意见。

        如果有更合适的景点，也请推荐可供替代的旅游景点。

        **你必须仅返回合法的 JSON 格式，不要输出任何其他内容。**

        返回的 JSON 必须始终包含以下字段，并且字段名称必须完全一致：

        - attraction_name
        - city
        - popularity_score
        - popularity_reasoning
        - visitor_rating
        - pros
        - cons
        - concierge_recommendation
        - alternative_suggestions
        """
    ),
    name="concierge_agent",
    default_options={"response_format": AttractionReview},
)


## 第四步：构建顺序工作流

SequentialBuilder 创建了一个工作流，其中：
1. **前台接待员** 接收用户输入并提供推荐
2. **礼宾员** 接收前台的推荐并进行专家审查
3. **输出** 包含原始推荐和专家审查


In [6]:
# ===== 第五部分：构建顺序工作流 =====
# 使用SequentialBuilder创建代理执行流程
workflow = SequentialBuilder(
    participants=[front_desk_agent, concierge_agent]  # 指定参与的代理及顺序
).build() # 构建工作流


display(HTML("""
<div style='padding: 20px; background: linear-gradient(135deg, #ff7043 0%, #ff5722 100%); color: white; border-radius: 8px; margin: 10px 0;'>
    <h3 style='margin: 0 0 15px 0;'>Sequential Workflow Built Successfully!</h3>
    <p style='margin: 0; line-height: 1.6;'>
        <strong>Flow:</strong><br>
        • User Input → <strong>Front Desk Agent</strong> (recommendation)<br>
        • Front Desk Output → <strong>Concierge Agent</strong> (review & rating)<br>
        • Final Output → Combined recommendation + expert review
    </p>
</div>
"""))

In [8]:
# ===== 第六部分：定义结果展示函数 =====
async def display_attraction_recommendation(city: str):
    """运行工作流并显示格式化结果"""

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>正在处理 {city} 的景点推荐</h3>
        <p style='margin: 0;'><strong>状态：</strong>运行顺序工作流中...</p>
    </div>
    """))

    # 执行工作流（用户提问 → 前台代理 → 礼宾代理）
    events = await workflow.run(f"我想参观{city}的一个景点。")
    outputs = events.get_outputs()

    if outputs:
        # 当前版本里 outputs[0] 是 AgentResponse，不是直接的消息列表。
        response = outputs[0]
        # 真正的消息列表位于 response.messages 中。
        messages: list[Message] = response.messages

        # Find front desk and concierge responses
        front_desk_response = None
        concierge_response = None

        for msg in messages:
            if msg.author_name == "front_desk_agent":
                front_desk_response = msg.text
            elif msg.author_name == "concierge_agent":
                concierge_response = msg.text

        display(HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; 
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h2 style='margin: 0 0 20px 0;'>推荐景点 {city}</h2>
            <p style='margin: 0; font-size: 14px; opacity: 0.9;'>由顺序代理工作流生成</p>
        </div>
        """))

        if front_desk_response:
            try:
                recommendation_data = AttractionRecommendation.model_validate_json(front_desk_response)
                display_front_desk_section(recommendation_data)
            except Exception as e:
                display(HTML(f"""
                <div style='padding: 15px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>前台响应错误解析:</strong> {str(e)}
                    <details><summary>原始响应</summary>{front_desk_response}</details>
                </div>
                 """))

        if concierge_response:
            try:
                review_data = AttractionReview.model_validate_json(concierge_response)
                display_concierge_section(review_data)
            except Exception as e:
                display(HTML(f"""
                <div style='padding: 15px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>礼宾响应错误解析:</strong> {str(e)}
                    <details><summary>原始响应</summary>{concierge_response}</details>
                </div>
                """))


def display_front_desk_section(data: AttractionRecommendation):
    """以美观格式显示前台推荐"""

    display(HTML(f"""
<div style='padding: 20px; background: #e3f2fd; border-radius: 8px; margin: 15px 0; border-left: 4px solid #2196f3;'>
    <h3 style='margin: 0 0 15px 0; color: #1976d2;'>🏨 前台推荐</h3>

    <div style='margin-bottom: 15px;'>
        <h4 style='margin: 0 0 8px 0; color: #333;'>{data.attraction_name}</h4>
        <span style='background: #2196f3; color: white; padding: 4px 8px; border-radius: 12px; font-size: 12px;'>{data.category}</span>
    </div>

    <div style='margin-bottom: 15px;'>
        <strong style='color: #333;'>景点介绍：</strong>
        {data.description}
    </div>

    <div style='margin-bottom: 10px;'>
        <strong style='color: #333;'>推荐理由：</strong>
        {data.why_recommended}
    </div>

    <div style='margin-bottom: 10px;'>
        <strong style='color: #333;'>建议游览时长：</strong>
        {data.recommended_duration}
    </div>

    <div>
        <strong style='color: #333;'>最佳游览时间：</strong>
        {data.best_time_to_visit}
    </div>
</div>
    """))


def display_concierge_section(data: AttractionReview):
    """以美观格式显示礼宾评审"""

    star_rating = "⭐" * int(data.visitor_rating) + "☆" * (5 - int(data.visitor_rating))
    popularity_bar = "🟩" * data.popularity_score + "⬜" * (10 - data.popularity_score)
    pros_list = "".join([f"<li style='color: #4caf50;'>✓ {pro}</li>" for pro in data.pros])
    cons_list = "".join([f"<li style='color: #f44336;'>✗ {con}</li>" for con in data.cons])
    alternatives_list = "".join([f"<li>{alt}</li>" for alt in data.alternative_suggestions])

    display(HTML(f"""
<div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
    <h3 style='margin: 0 0 15px 0; color: #f57c00;'>🎩 礼宾部专家评审</h3>

    <div style='margin-bottom: 15px;'>
        <strong style='color: #333;'>景点热度评分：</strong>
        {data.popularity_score}/10 {popularity_bar}
    </div>

    <div style='margin-bottom: 15px;'>
        <strong style='color: #333;'>游客评分：</strong>
        {data.visitor_rating}/5 {star_rating}
    </div>

    <div style='margin-bottom: 15px;'>
        <strong style='color: #333;'>评分依据：</strong>
        {data.popularity_reasoning}
    </div>

    <div style='margin-bottom: 15px;'>
        <strong style='color: #333;'>专家建议：</strong>
        {data.concierge_recommendation}
    </div>

    <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 20px;'>
        <div>
            <strong style='color: #333;'>优点：</strong>
            <ul style='margin: 8px 0;'>{pros_list}</ul>
        </div>

        <div>
            <strong style='color: #333;'>不足：</strong>
            <ul style='margin: 8px 0;'>{cons_list}</ul>
        </div>
    </div>

    <div style='margin-top: 15px;'>
        <strong style='color: #333;'>其他推荐景点：</strong>
        <ul style='margin: 8px 0;'>{alternatives_list}</ul>
    </div>
</div>
    """))


# Test with Stockholm
await display_attraction_recommendation("斯德哥尔摩")


## 第8步：工作流程分析 - 理解顺序流程

让我们来研究信息如何在代理之间流动，并分析对话历史记录。


In [10]:
async def analyze_sequential_flow(city: str):
    """Analyze the sequential flow between agents."""

    display(HTML(f"""
        <div style='padding: 20px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 8px; margin: 20px 0;'>
            <h3 style='margin: 0 0 10px 0; color: #7b1fa2;'>🔄 {city} 多智能体协同分析</h3>
            <p style='margin: 0;'>正在分析各智能体之间的协作流程及信息传递...</p>
        </div>
    """))

    # Run the workflow
    events = await workflow.run(f"我想参观{city}的一个景点。")
    outputs = events.get_outputs()

    if outputs:
        # 当前版本里 outputs[0] 是 AgentResponse，不是直接可迭代的消息列表。
        response = outputs[0]
        messages: list[Message] = response.messages

        display(HTML(f"""
        <div style='padding: 25px; background: #f3e5f5; border-radius: 12px; margin: 20px 0;'>
            <h2 style='margin: 0 0 20px 0; color: #7b1fa2;'>💬 对话流程分析</h2>
        </div>
        """))

        # Display each message in the sequence
        for i, msg in enumerate(messages, 1):
            role_color = {
                "user": "#2196f3",
                "front_desk_agent": "#4caf50",
                "concierge_agent": "#ff9800"
            }.get(msg.author_name or "user", "#666666")

            role_name = {
                "user": "👤 User",
                "front_desk_agent": "🏨 前台代理",
                "concierge_agent": "🎩 礼宾代理"
            }.get(msg.author_name or "user", "Unknown")

            content_preview = msg.text[:200] + "..." if len(msg.text) > 200 else msg.text
            display(HTML(f"""
            <div style='padding: 15px; background: white; border-left: 4px solid {role_color}; border-radius: 4px; margin: 10px 0; box-shadow: 0 2px 4px rgba(0,0,0,0.1);'>
                <div style='display: flex; align-items: center; margin-bottom: 10px;'>
                    <span style='font-weight: bold; color: {role_color}; margin-right: 10px;'>Step {i}:</span>
                    <span style='font-weight: bold; color: {role_color};'>{role_name}</span>
                </div>
                <div style='color: #555; font-size: 14px; line-height: 1.4;'>
                    {content_preview}
                </div>
            </div>
            """))

        # Analyze the flow
        display(HTML(f"""
            <div style='padding: 20px; background: linear-gradient(135deg, #9c27b0 0%, #673ab7 100%); color: white; border-radius: 8px; margin: 20px 0;'>
                <h3 style='margin: 0 0 15px 0;'>📊 流程分析总结</h3>

                <ul style='margin: 0; padding-left: 20px; line-height: 1.6;'>
                    <li><strong>消息总数：</strong>{len(messages)}</li>
                    <li><strong>参与智能体：</strong>2 个（前台接待员 + 礼宾部专家）</li>
                    <li><strong>流程模式：</strong>线性顺序流程（用户 → 智能体 1 → 智能体 2）</li>
                    <li><strong>信息交接：</strong>前台的推荐结果作为礼宾部专家的输入</li>
                    <li><strong>输出质量：</strong>通过专家评审与评分进一步提升推荐质量</li>
                </ul>
            </div>
        """))


# Analyze the flow for Barcelona
await analyze_sequential_flow("巴塞罗纳")



---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于重要信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
